# NOTEBOOK 1: XAI HEATMAP GENERATION FACTORY
# ISIC 2019 Skin Lesion Classification — Explainability Pipeline
# PURPOSE : Generate and save Grad-CAM++ (DenseNet, Swin) and Score-CAM (EfficientNet)
#           heatmaps for the validation/test set. Results are persisted as .npy files
#           so downstream notebooks never have to re-run GPU inference.

In [1]:
import os
import numpy as np

OUTPUT_DIR = '/kaggle/working/xai_outputs'
BINARY_OUT = os.path.join(OUTPUT_DIR, 'binary')
MULTI_OUT  = os.path.join(OUTPUT_DIR, 'multiclass')

print("=" * 50)
print("             XAI FACTORY AUDIT REPORT          ")
print("=" * 50)

# 1. Check File Counts
print("\n[1/3] Checking Saved Matrix Counts:")
binary_folders = ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin', 'heatmaps_convnext', 'heatmaps_resnet', 'heatmaps_ensemble']
severity_folders = ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin']

print("  ── Binary Run Output (Notebook 1 - Part 1) ──")
for folder in binary_folders:
    path = os.path.join(BINARY_OUT, folder)
    count = len([f for f in os.listdir(path) if f.endswith('.npy')]) if os.path.exists(path) else 0
    print(f"    {folder:<25}: {count} files expected ~5067")

print("\n  ── Multiclass Severity Output (Notebook 1 - Part 2) ──")
for folder in severity_folders:
    path = os.path.join(MULTI_OUT, folder)
    count = len([f for f in os.listdir(path) if f.endswith('.npy')]) if os.path.exists(path) else 0
    print(f"    {folder:<25}: {count} files expected ~9340")

# 2. Check Meta-Indices
print("\n[2/3] Checking Metadata Index CSVs:")
index_path = os.path.join(BINARY_OUT, 'heatmap_index.csv')
if os.path.exists(index_path):
    print(f"    ✓ heatmap_index.csv found at {index_path}")
    import pandas as pd
    df_check = pd.read_csv(index_path)
    print(f"    ✓ Registered mapped rows: {len(df_check)}")
else:
    print("    ✗ Critical Warning: heatmap_index.csv is missing from binary output.")

# 3. Structural Matrix Integrity Check
print("\n[3/3] Testing Random Matrix Integrity:")
try:
    # Pick a random sample from ensemble
    sample_dir = os.path.join(BINARY_OUT, 'heatmaps_ensemble')
    sample_file = [f for f in os.listdir(sample_dir) if f.endswith('.npy')][0]
    matrix = np.load(os.path.join(sample_dir, sample_file))
    
    print(f"    ✓ Loaded sample matrix: {sample_file}")
    print(f"    ✓ Spatial Dimensions   : {matrix.shape} (Should be 224x224 after fusion)")
    print(f"    ✓ Value Normalization  : Min={matrix.min():.2f}, Max={matrix.max():.2f} (Should be 0.00 to 1.00)")
    print("\n[STATUS]: Everything is completely intact. Safe to compress and transition downstream.")
except Exception as e:
    print(f"    ✗ Error testing file matrix integrity: {str(e)}")
print("=" * 50)

             XAI FACTORY AUDIT REPORT          

[1/3] Checking Saved Matrix Counts:
  ── Binary Run Output (Notebook 1 - Part 1) ──
    heatmaps_densenet        : 0 files expected ~5067
    heatmaps_efficientnet    : 0 files expected ~5067
    heatmaps_swin            : 0 files expected ~5067
    heatmaps_convnext        : 0 files expected ~5067
    heatmaps_resnet          : 0 files expected ~5067
    heatmaps_ensemble        : 0 files expected ~5067

  ── Multiclass Severity Output (Notebook 1 - Part 2) ──
    heatmaps_densenet        : 0 files expected ~9340
    heatmaps_efficientnet    : 0 files expected ~9340
    heatmaps_swin            : 0 files expected ~9340

[2/3] Checking Metadata Index CSVs:
    ✗ Critical Warning: heatmap_index.csv is missing from binary output.

[3/3] Testing Random Matrix Integrity:
    ✗ Error testing file matrix integrity: [Errno 2] No such file or directory: '/kaggle/working/xai_outputs/binary/heatmaps_ensemble'


In [2]:
# ── CELL 1 : Installs & Imports ──────────────────────────────────────────────
import os, gc, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")

Device : cuda


In [3]:
# ── CELL 2 : Configuration ────────────────────────────────────────────────────
# ── Input paths ──────────────────────────────────────────────────────────────
INPUT_CSV_DIR   = '/kaggle/input/notebooks/nandini143/data-preparation'
MODEL_DIR       = '/kaggle/input/notebooks/nandini143'

# ── Binary classification checkpoint paths ────────────────────────────────────
DENSENET_CKPT   = '/kaggle/input/notebooks/nandini143/binary-densenet-121/densenet121_best.pth'
EFFNET_CKPT     = '/kaggle/input/notebooks/nandini143/binary-efficientnetv2s/efficientnetv2s_best.pth'
SWIN_CKPT       = '/kaggle/input/notebooks/nandini143/binary-swin-tiny-transformer/swin_tiny_best.pth'
CONVNEXT_CKPT   = '/kaggle/input/notebooks/nandini143/binary-convnext/convnext_tiny_best.pth'
RESNET_CKPT     = '/kaggle/input/notebooks/nandini143/binary-resnet-200d/resnet200d_best.pth'

# ── Multiclass (Severity) checkpoint paths ────────────────────────────────────
SEVERITY_DENSENET_CKPT  = '/kaggle/input/notebooks/nandini143/densenet-121-ordinal-severity/densenet_severity/densenet_severity_best.pth'
SEVERITY_EFFNET_CKPT    = '/kaggle/input/notebooks/nandini143/efficientnetv2-ordinal-severity/effnet_severity/effnet_severity_best.pth'
SEVERITY_SWIN_CKPT      = '/kaggle/input/notebooks/nandini143/swin-tiny-ordinal-severity/swin_severity/swin_severity_best.pth'

# ── Multiclass severity CSV (for image IDs & metadata) ───────────────────────
SEVERITY_CSV    = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/severity_train_folds.csv'
SEVERITY_IMGDIR = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/preprocessed_images'

# ── CORAL config: 4 severity classes (AKIEC, BCC, SCC, MEL) → 3 binary outputs
SEVERITY_NUM_OUTPUTS = 3
SEVERITY_META_DIM    = 9    # 3 (age_norm, sex_male, sex_female) + 6 site dummies (typical ISIC)
STAGE_NAMES          = {0: 'AK', 1: 'BCC', 2: 'SCC', 3: 'MEL'}

# ── Image sizes ───────────────────────────────────────────────────────────────
IMG_SIZE_CNN    = 384
IMG_SIZE_SWIN   = 224
BATCH_SIZE      = 1    # MUST be 1 for accurate XAI generation

# ── Path replacement (for loading images from Kaggle datasets) ────────────────
OLD_PATH        = '/kaggle/working/preprocessed/'
NEW_PATH        = f'{INPUT_CSV_DIR}/preprocessed/'

# ── Normalization ─────────────────────────────────────────────────────────────
IMAGENET_MEAN   = [0.485, 0.456, 0.406]
IMAGENET_STD    = [0.229, 0.224, 0.225]

# ── Output directories: separated for binary vs multiclass heatmaps ───────────
OUTPUT_DIR      = '/kaggle/working/xai_outputs'
BINARY_OUT      = os.path.join(OUTPUT_DIR, 'binary')
MULTI_OUT       = os.path.join(OUTPUT_DIR, 'multiclass')

for sub in ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin', 'heatmaps_convnext', 'heatmaps_resnet', 'heatmaps_ensemble']:
    os.makedirs(os.path.join(BINARY_OUT, sub), exist_ok=True)
    os.makedirs(os.path.join(MULTI_OUT, sub),  exist_ok=True)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config ready.')
print(f'  Binary   outputs -> {BINARY_OUT}')
print(f'  Severity outputs -> {MULTI_OUT}')

Config ready.
  Binary   outputs -> /kaggle/working/xai_outputs/binary
  Severity outputs -> /kaggle/working/xai_outputs/multiclass


In [4]:
# ── CELL 3a : Binary Model Architectures (CORRECTED) ───────────────────────
class TabularMLP_ConvNeXt(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=32, out_dim=32, dropout=0.4):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden_dim), torch.nn.BatchNorm1d(hidden_dim), torch.nn.GELU(),
            torch.nn.Dropout(dropout), torch.nn.Linear(hidden_dim, out_dim), torch.nn.GELU()
        )
    def forward(self, x): return self.net(x)

# ── FIXED: Added MultiModalWrappers for DenseNet and EfficientNet to match checkpoint ──
class MultiModalDenseNet(torch.nn.Module):
    def __init__(self, backbone_name='densenet121', meta_dim=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        img_feat_dim = self.backbone.num_features
        self.meta_mlp = TabularMLP_ConvNeXt(meta_dim, 32, 32, 0.4)
        fusion_dim = img_feat_dim + 32
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.4), torch.nn.Linear(fusion_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1)
        )
    def forward(self, images, meta=None):
        img_feats = self.backbone(images)
        meta_feats = self.meta_mlp(meta) if meta is not None else self.meta_mlp(torch.zeros(images.size(0), 11, device=images.device))
        return self.head(torch.cat([img_feats, meta_feats], dim=1))

class MultiModalEfficientNet(torch.nn.Module):
    def __init__(self, backbone_name='tf_efficientnetv2_s', meta_dim=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg', drop_path_rate=0.2)
        img_feat_dim = self.backbone.num_features
        self.meta_mlp = TabularMLP_ConvNeXt(meta_dim, 32, 32, 0.4)
        fusion_dim = img_feat_dim + 32
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.4), torch.nn.Linear(fusion_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1)
        )
    def forward(self, images, meta=None):
        img_feats = self.backbone(images)
        meta_feats = self.meta_mlp(meta) if meta is not None else self.meta_mlp(torch.zeros(images.size(0), 11, device=images.device))
        return self.head(torch.cat([img_feats, meta_feats], dim=1))

class MultiModalConvNeXt(torch.nn.Module):
    def __init__(self, backbone_name='convnext_tiny', meta_dim=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        img_feat_dim = self.backbone.num_features
        self.meta_mlp = TabularMLP_ConvNeXt(meta_dim, 32, 32, 0.4)
        fusion_dim = img_feat_dim + 32
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.4), torch.nn.Linear(fusion_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1)
        )
    def forward(self, images, meta=None):
        img_feats = self.backbone(images)
        meta_feats = self.meta_mlp(meta) if meta is not None else self.meta_mlp(torch.zeros(images.size(0), 11, device=images.device))
        return self.head(torch.cat([img_feats, meta_feats], dim=1))

class MultiModalResNet(torch.nn.Module):
    def __init__(self, model_name='resnet200d', num_meta_features=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features
        self.meta_branch = torch.nn.Sequential(
            torch.nn.Linear(num_meta_features, 64), torch.nn.BatchNorm1d(64), torch.nn.SiLU(),
            torch.nn.Dropout(0.2), torch.nn.Linear(64, 32), torch.nn.BatchNorm1d(32), torch.nn.SiLU()
        )
        self.head = torch.nn.Sequential(
            torch.nn.Linear(in_features + 32, 256), torch.nn.BatchNorm1d(256), torch.nn.SiLU(),
            torch.nn.Dropout(0.4), torch.nn.Linear(256, 1)
        )
    def forward(self, image, meta=None):
        img_features = self.backbone(image)
        meta_features = self.meta_branch(meta) if meta is not None else self.meta_branch(torch.zeros(image.size(0), 11, device=image.device))
        return self.head(torch.cat([img_features, meta_features], dim=1))

class MultiModalSwin(torch.nn.Module):
    def __init__(self, backbone_name='swin_tiny_patch4_window7_224', meta_dim=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, drop_path_rate=0.2)
        img_feat_dim = self.backbone.num_features 
        self.meta_mlp = TabularMLP_ConvNeXt(meta_dim, 32, 32, 0.4)
        fusion_dim = img_feat_dim + 32 
        
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.4), torch.nn.Linear(fusion_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1) 
        )

    def forward(self, images, meta=None):
        img_feats = self.backbone(images)
        if len(img_feats.shape) == 3:
            img_feats = img_feats.mean(dim=1)
            
        meta_feats = self.meta_mlp(meta) if meta is not None else self.meta_mlp(torch.zeros(images.size(0), 11, device=images.device))
        return self.head(torch.cat([img_feats, meta_feats], dim=1))

# ── Model Builders ──────────────────────────────────────────────────────────
def build_densenet121(pretrained=False):
    return MultiModalDenseNet(pretrained=pretrained)
def build_efficientnetv2s(pretrained=False):
    return MultiModalEfficientNet(pretrained=pretrained)
def build_swin_tiny(pretrained=False):
    return MultiModalSwin(pretrained=pretrained)
def build_convnext_tiny(pretrained=False):
    return MultiModalConvNeXt(pretrained=pretrained)
def build_resnet200d(pretrained=False):
    return MultiModalResNet(pretrained=pretrained)

def load_model(builder_fn, ckpt_path, device):
    model = builder_fn()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state_dict)
    model.to(device).eval()
    return model

# ── Target Layer Finder ─────────────────────────────────────────────────────
def find_last_conv_layer(model, model_name):
    # Dynamically extract backbone if it's a multimodal wrapper
    base_model = getattr(model, 'backbone', model)
    
    if 'densenet' in model_name:
        return base_model.features.denseblock4
    elif 'efficientnet' in model_name:
        return base_model.conv_head
    elif 'swin' in model_name:
        if hasattr(base_model, 'layers'):
            return base_model.layers[-1].blocks[-1]
    elif 'convnext' in model_name:
        return base_model.stages[-1].blocks[-1]
    elif 'resnet' in model_name:
        return base_model.layer4[-1]
    return None

In [5]:
# ── CELL 3b : Multiclass (Severity) Model Architectures ──────────────────────
class TabularMLP(nn.Module):
    """Processes clinical metadata (age, sex, site) before fusion with image features."""
    def __init__(self, in_dim: int, hidden_dim: int = 64, out_dim: int = 32,
                 dropout: float = 0.2, use_gelu: bool = True):
        super().__init__()
        act = nn.GELU() if use_gelu else nn.ReLU(inplace=True)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), act,
            nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim), act,
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class MetadataSeverityModel(nn.Module):
    """DenseNet-121 and EfficientNetV2-S severity models."""
    def __init__(self, model_name: str, num_outputs: int, meta_dim: int,
                 pretrained: bool = False, drop_rate: float = 0.3,
                 drop_path_rate: float = 0.0, use_gelu: bool = True):
        super().__init__()
        kwargs = dict(pretrained=pretrained, num_classes=0, drop_rate=drop_rate)
        if drop_path_rate > 0:
            kwargs['drop_path_rate'] = drop_path_rate
        self.backbone  = timm.create_model(model_name, **kwargs)
        feat_dim       = self.backbone.num_features
        self.use_meta  = meta_dim > 0
        if self.use_meta:
            self.meta_mlp = TabularMLP(meta_dim, hidden_dim=64, out_dim=32, dropout=0.2, use_gelu=use_gelu)
            fusion_dim = feat_dim + 32
        else:
            fusion_dim = feat_dim
            
        norm = nn.BatchNorm1d(fusion_dim) if 'densenet' in model_name else nn.LayerNorm(fusion_dim)
        act  = nn.ReLU(inplace=True) if 'densenet' in model_name else nn.GELU()
        self.head = nn.Sequential(
            norm, nn.Linear(fusion_dim, 512 if 'densenet' in model_name else 256), act,
            nn.Dropout(drop_rate), nn.Linear(512 if 'densenet' in model_name else 256, num_outputs),
        )
        
    def forward(self, images: torch.Tensor, meta: torch.Tensor = None):
        feat = self.backbone(images)
        if self.use_meta and meta is not None:
            feat = torch.cat([feat, self.meta_mlp(meta)], dim=1)
        return self.head(feat)

class SwinSeverityModel(nn.Module):
    """Swin-Tiny severity model with LayerNorm head and metadata fusion."""
    def __init__(self, model_name: str, num_outputs: int, meta_dim: int,
                 pretrained: bool = False, drop_rate: float = 0.2,
                 drop_path_rate: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, drop_rate=drop_rate, drop_path_rate=drop_path_rate
        )
        feat_dim      = self.backbone.num_features 
        self.use_meta = meta_dim > 0
        if self.use_meta:
            self.meta_mlp = TabularMLP(meta_dim, 64, 32, 0.2, use_gelu=True)
            fusion_dim = feat_dim + 32
        else:
            fusion_dim = feat_dim
            
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, 512), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, 128), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(128, num_outputs),
        )
        
    def forward(self, images: torch.Tensor, meta: torch.Tensor = None):
        feat = self.backbone(images)
        if self.use_meta and meta is not None:
            # FIXED: Corrected self.meta_mlp(feat) to self.meta_mlp(meta)
            feat = torch.cat([feat, self.meta_mlp(meta)], dim=1) 
        return self.head(feat)

# ── Severity model builders ────────────────────────────────────────────────────
def build_severity_densenet(meta_dim: int = SEVERITY_META_DIM, pretrained: bool = False):
    return MetadataSeverityModel(
        'densenet121', SEVERITY_NUM_OUTPUTS, meta_dim,
        pretrained=pretrained, drop_rate=0.3, drop_path_rate=0.0, use_gelu=False
    )
def build_severity_efficientnet(meta_dim: int = SEVERITY_META_DIM, pretrained: bool = False):
    return MetadataSeverityModel(
        'tf_efficientnetv2_s', SEVERITY_NUM_OUTPUTS, meta_dim,
        pretrained=pretrained, drop_rate=0.3, drop_path_rate=0.2, use_gelu=True
    )
def build_severity_swin(meta_dim: int = SEVERITY_META_DIM, pretrained: bool = False):
    return SwinSeverityModel(
        'swin_tiny_patch4_window7_224', SEVERITY_NUM_OUTPUTS, meta_dim,
        pretrained=pretrained, drop_rate=0.2, drop_path_rate=0.2
    )
    
def load_severity_model(builder_fn, ckpt_path, device, meta_dim=SEVERITY_META_DIM):
    model = builder_fn(meta_dim=meta_dim)
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get('model_state', ckpt.get('model_state_dict', ckpt))
    model.load_state_dict(state, strict=False)
    model.to(device).eval()
    return model

def coral_logits_to_label(logits: torch.Tensor) -> torch.Tensor:
    return (torch.sigmoid(logits) > 0.5).sum(dim=1)

def find_severity_target_layer(model, model_type: str):
    if model_type == 'densenet':
        return model.backbone.features.denseblock4
    elif model_type == 'efficientnet':
        return model.backbone.conv_head
    elif model_type == 'swin':
        return model.backbone.layers[-1].blocks[-1]
    return None

print('Severity model architectures loaded.')
print(f'  Classes : MetadataSeverityModel (DenseNet, EfficientNet), SwinSeverityModel')
print(f'  Builders: build_severity_densenet | build_severity_efficientnet | build_severity_swin')
print(f'  CORAL   : {SEVERITY_NUM_OUTPUTS} outputs -> severity class 0/1/2/3 (AKIEC/BCC/SCC/MEL)')

Severity model architectures loaded.
  Classes : MetadataSeverityModel (DenseNet, EfficientNet), SwinSeverityModel
  Builders: build_severity_densenet | build_severity_efficientnet | build_severity_swin
  CORAL   : 3 outputs -> severity class 0/1/2/3 (AKIEC/BCC/SCC/MEL)


In [6]:
# ── CELL 4 : Dataset ──────────────────────────────────────────────────────────
class XAIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        img_t = self.transform(img)
        return img_t, str(row.get('image_id', idx))

def get_transforms(img_size):
    return T.Compose([
        T.Resize((img_size, img_size)), T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

val_df = pd.read_csv(f'{INPUT_CSV_DIR}/binary_val.csv')
test_df = pd.read_csv(f'{INPUT_CSV_DIR}/binary_test.csv')
eval_df = pd.concat([val_df, test_df], ignore_index=True)
eval_df['image_path'] = eval_df['image_path'].str.replace(OLD_PATH, NEW_PATH, regex=False)

In [7]:
# ─── CELL 5 : XAI Engine — FIXED GradCAM++, Score-CAM & Attention Rollout ───
import torch.nn.functional as F
import numpy as np

class RobustGradCAMPlusPlus:
    """
    Fixed GradCAM++ with Binary Logit Flip.
    For BINARY models (1 output node):
      - If model predicts Malignant (prob >= 0.5): backprop on the Malignant logit.
      - If model predicts Benign   (prob  < 0.5): backprop on NEGATED logit.
        This forces gradients to highlight BENIGN features instead of empty maps.
    For MULTICLASS / CORAL models: uses target_class index as before.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients   = None
        self._fwd_hook = target_layer.register_forward_hook(self._save_activation)
        self._bwd_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _reshape_transform(self, x):
        if len(x.shape) == 3:
            B, L, C = x.shape
            H = W = int(np.sqrt(L))
            return x.transpose(1, 2).reshape(B, C, H, W)
        return x

    def _save_activation(self, module, input, output):
        self.activations = self._reshape_transform(output.detach())

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = self._reshape_transform(grad_output[0].detach())

    def generate(self, input_tensor, meta=None, target_class=None, n_smooth=5, noise_sigma=0.02):
        smooth_cams = []
        for _ in range(n_smooth):
            noise       = torch.randn_like(input_tensor) * noise_sigma
            noisy_input = (input_tensor + noise).detach().requires_grad_(True)
            self.model.zero_grad()

            if target_class is None:
                # BINARY MODEL — apply logit flip for Benign images
                logits = self.model(noisy_input, meta)
                prob   = torch.sigmoid(logits).item()
                score  = logits.squeeze() if prob >= 0.5 else -logits.squeeze()
            else:
                # MULTICLASS / CORAL MODEL
                meta_tmp  = meta if meta is not None else torch.zeros(1, SEVERITY_META_DIM, device=noisy_input.device)
                logits    = self.model(noisy_input, meta_tmp)
                logit_idx = max(int(target_class) - 1, 0)
                score     = logits[0, logit_idx]

            score.backward()
            acts, grads = self.activations, self.gradients

            grads_sq    = grads ** 2
            grads_cu    = grads ** 3
            alpha_num   = grads_sq
            alpha_denom = 2 * grads_sq + (acts * grads_cu).sum(dim=(2, 3), keepdim=True) + 1e-8
            alpha   = alpha_num / alpha_denom
            weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)
            cam     = (weights * acts).sum(dim=1)
            cam     = F.relu(cam).squeeze(0).cpu().numpy()

            if cam.max() > 0:
                cam = (cam - cam.min()) / (cam.max() - cam.min())
            smooth_cams.append(cam)

        avg_cam = np.mean(smooth_cams, axis=0)
        if avg_cam.max() > 0:
            avg_cam = (avg_cam - avg_cam.min()) / (avg_cam.max() - avg_cam.min())
        return avg_cam

    def remove_hooks(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()


class RobustScoreCAM:
    """
    Fixed Score-CAM for EfficientNetV2-S (gradient-free).
    
    PREVIOUS BUG: cam += (score - base_prob) * mask
      Masking the tumour LOWERS score, so score - base_prob is NEGATIVE,
      then ReLU zeroes it → blank heatmap.
    
    FIX: Use class-aware weight:
      Malignant image → weight = score         (high when tumour is visible)
      Benign   image  → weight = (1 - score)   (high when benign features visible)
    """
    def __init__(self, model, target_layer, top_k=64):
        self.model, self.top_k = model, top_k
        self.activations = None
        self._fwd_hook = target_layer.register_forward_hook(self._save)

    def _save(self, module, input, output):
        self.activations = output.detach()

    @torch.no_grad()
    def generate(self, input_tensor, meta=None, target_class=None):
        B, C_in, H, W = input_tensor.shape

        if target_class is None:
            logits       = self.model(input_tensor, meta)
            base_prob    = torch.sigmoid(logits).item()
            is_malignant = (base_prob >= 0.5)
        else:
            meta_tmp     = meta if meta is not None else torch.zeros(1, SEVERITY_META_DIM, device=input_tensor.device)
            logits       = self.model(input_tensor, meta_tmp)
            logit_idx    = max(int(target_class) - 1, 0)
            base_prob    = torch.sigmoid(logits[0, logit_idx]).item()
            is_malignant = True

        acts    = self.activations
        norms   = acts.std(dim=(2, 3)).squeeze()
        top_idx = norms.topk(min(self.top_k, acts.shape[1])).indices
        cam     = torch.zeros(H, W, device=input_tensor.device)

        for ch in top_idx:
            mask    = acts[0, ch, :, :].unsqueeze(0).unsqueeze(0)
            mask    = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
            mask_up = F.interpolate(mask, size=(H, W), mode='bilinear', align_corners=False)

            masked_input = input_tensor * mask_up
            if target_class is None:
                score  = torch.sigmoid(self.model(masked_input, meta)).item()
                weight = score if is_malignant else (1.0 - score)
            else:
                out    = self.model(masked_input, meta_tmp)
                weight = torch.sigmoid(out[0, logit_idx]).item()

            cam += weight * mask_up.squeeze()

        cam = F.relu(cam).cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

    def remove_hooks(self):
        self._fwd_hook.remove()
        self.model = None


class AttentionRolloutSwin:
    """
    Attention Rollout for Swin-Tiny Transformer.
    
    WHY: GradCAM++ gave 0.00% Drop-Hit for Swin. Shifted-Window attention
         scrambles spatial gradients across window boundaries, making GradCAM
         mathematically meaningless for Swin.
    
    FIX: Attention Rollout traces the actual attention flow through ALL
         transformer layers by multiplying attention matrices, correctly
         identifying which spatial patches the model focused on.
    """
    def __init__(self, model):
        self.model  = model
        self.attns  = []
        self._hooks = []
        self._register_hooks()

    def _register_hooks(self):
        for name, layer in self.model.named_modules():
            if 'WindowAttention' in type(layer).__name__:
                h = layer.register_forward_hook(self._capture_attn)
                self._hooks.append(h)

    def _capture_attn(self, module, input, output):
        # Attempt to capture stored attention weights from the module
        if hasattr(module, 'attn') and module.attn is not None:
            try:
                self.attns.append(module.attn.detach().cpu().float())
            except Exception:
                pass

    @torch.no_grad()
    def generate(self, input_tensor, meta=None, target_class=None):
        self.attns = []
        B, C, H, W = input_tensor.shape
        _ = self.model(input_tensor, meta)

        if len(self.attns) == 0:
            # Fallback: uniform importance map if no attention weights captured
            return np.ones((H, W), dtype=np.float32) * 0.5

        result = None
        for attn in self.attns:
            attn_avg = attn.mean(dim=1)  # (B, N, N) — average over heads
            eye      = torch.eye(attn_avg.shape[-1])
            attn_aug = attn_avg + eye
            attn_aug = attn_aug / attn_aug.sum(dim=-1, keepdim=True)

            if result is None:
                result = attn_aug
            else:
                try:
                    result = torch.bmm(result, attn_aug)
                except Exception:
                    result = attn_aug  # shape mismatch fallback

        # Take attention from first token to all spatial patches
        mask = result[0, 0, :].numpy()
        grid_size = int(np.sqrt(len(mask)))
        if grid_size * grid_size != len(mask):
            grid_size = max(1, int(len(mask) ** 0.5))
            mask = mask[:grid_size * grid_size]
        mask = mask.reshape(grid_size, grid_size)
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)

        import cv2
        mask_up = cv2.resize(mask.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
        return mask_up

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks = []


print('XAI Engine v2 ready.')
print('  RobustGradCAMPlusPlus : binary (logit-flip) + multiclass')
print('  RobustScoreCAM        : EfficientNet (class-aware Score-CAM)')
print('  AttentionRolloutSwin  : Swin-Tiny (replaces broken GradCAM++)')


XAI Engine v2 ready.
  RobustGradCAMPlusPlus : binary (logit-flip) + multiclass
  RobustScoreCAM        : EfficientNet (class-aware Score-CAM)
  AttentionRolloutSwin  : Swin-Tiny (replaces broken GradCAM++)


In [8]:
# ── CELL 6 : Heatmap Saving Utility ──────────────────────────────────────────
def save_heatmap(heatmap, out_dir, image_id):
    np.save(os.path.join(out_dir, f"{image_id}.npy"), heatmap.astype(np.float32))

def upsample_heatmap(heatmap, h, w):
    t = torch.from_numpy(heatmap).float().unsqueeze(0).unsqueeze(0)
    return F.interpolate(t, size=(h, w), mode='bilinear', align_corners=False).squeeze().numpy()

In [9]:
# ── CELL 7 : Generate Swin-Tiny Binary Heatmaps (FIXED: Attention Rollout) ──
print("\n[1/5] Generating Swin-Tiny Heatmaps (Attention Rollout)...")
swin_model = load_model(build_swin_tiny, SWIN_CKPT, DEVICE)

# FIX: Use AttentionRolloutSwin. GradCAM++ had 0.00% Drop-Hit on Swin because
# Shifted-Window attention scrambles spatial gradients across window borders.
# Attention Rollout traces actual attention flow through all transformer layers.
ar_swin = AttentionRolloutSwin(swin_model)

out_dir = os.path.join(BINARY_OUT, 'heatmaps_swin')
os.makedirs(out_dir, exist_ok=True)
loader  = DataLoader(XAIDataset(eval_df, get_transforms(IMG_SIZE_SWIN)), batch_size=1)

for imgs, img_ids in tqdm(loader, desc="Swin-Tiny"):
    heatmap = ar_swin.generate(imgs.to(DEVICE))
    save_heatmap(heatmap, out_dir, img_ids[0])

ar_swin.remove_hooks()
del swin_model; gc.collect(); torch.cuda.empty_cache()
print("Swin-Tiny heatmaps complete.")



[1/5] Generating Swin-Tiny Heatmaps (Attention Rollout)...


Swin-Tiny: 100%|██████████| 5067/5067 [01:49<00:00, 46.26it/s]


Swin-Tiny heatmaps complete.


In [10]:
# ── CELL 8 : Generate DenseNet Grad-CAM++ ────────────────────────────────────
print("\n[2/5] Generating DenseNet-121 Heatmaps...")
dense_model = load_model(build_densenet121, DENSENET_CKPT, DEVICE)
layer_dense = find_last_conv_layer(dense_model, 'densenet121')
gcam_dense = RobustGradCAMPlusPlus(dense_model, layer_dense)

loader = DataLoader(XAIDataset(eval_df, get_transforms(IMG_SIZE_CNN)), batch_size=1)

for imgs, img_ids in tqdm(loader, desc="DenseNet"):
    save_heatmap(gcam_dense.generate(imgs.to(DEVICE)), os.path.join(BINARY_OUT, 'heatmaps_densenet'), img_ids[0])

gcam_dense.remove_hooks(); del dense_model; gc.collect(); torch.cuda.empty_cache()


[2/5] Generating DenseNet-121 Heatmaps...


DenseNet: 100%|██████████| 5067/5067 [21:16<00:00,  3.97it/s]


In [11]:
# ── CELL 9 : Generate EfficientNet Score-CAM ─────────────────────────────────
print("\n[3/5] Generating EfficientNetV2-S Heatmaps...")
eff_model = load_model(build_efficientnetv2s, EFFNET_CKPT, DEVICE)
layer_eff = find_last_conv_layer(eff_model, 'efficientnet')
scam_eff = RobustScoreCAM(eff_model, layer_eff)

loader = DataLoader(XAIDataset(eval_df, get_transforms(IMG_SIZE_CNN)), batch_size=1)

for imgs, img_ids in tqdm(loader, desc="EfficientNet"):
    save_heatmap(scam_eff.generate(imgs.to(DEVICE)), os.path.join(BINARY_OUT, 'heatmaps_efficientnet'), img_ids[0])

scam_eff.remove_hooks(); del eff_model; gc.collect(); torch.cuda.empty_cache()


[3/5] Generating EfficientNetV2-S Heatmaps...


EfficientNet: 100%|██████████| 5067/5067 [1:35:30<00:00,  1.13s/it]


In [12]:
# ── CELL 10 : Generate ConvNeXt-Tiny Heatmaps ─────────────────────────────────
print("\n[4/5] Generating ConvNeXt-Tiny Heatmaps...")
conv_model = load_model(build_convnext_tiny, CONVNEXT_CKPT, DEVICE)
layer_conv = find_last_conv_layer(conv_model, 'convnext')
gcam_conv = RobustGradCAMPlusPlus(conv_model, layer_conv)

loader = DataLoader(XAIDataset(eval_df, get_transforms(IMG_SIZE_CNN)), batch_size=1)

for imgs, img_ids in tqdm(loader, desc="ConvNeXt"):
    save_heatmap(gcam_conv.generate(imgs.to(DEVICE)), os.path.join(BINARY_OUT, 'heatmaps_convnext'), img_ids[0])

gcam_conv.remove_hooks(); del conv_model; gc.collect(); torch.cuda.empty_cache()


[4/5] Generating ConvNeXt-Tiny Heatmaps...


ConvNeXt: 100%|██████████| 5067/5067 [24:00<00:00,  3.52it/s]


In [13]:
# ── CELL 11 : Generate ResNet-200D Heatmaps ─────────────────────────────────
print("\n[5/5] Generating ResNet-200D Heatmaps...")
resnet_model = load_model(build_resnet200d, RESNET_CKPT, DEVICE)
layer_resnet = find_last_conv_layer(resnet_model, 'resnet')
gcam_resnet = RobustGradCAMPlusPlus(resnet_model, layer_resnet)

# ResNet-200D was trained on 224
loader = DataLoader(XAIDataset(eval_df, get_transforms(224)), batch_size=1)

for imgs, img_ids in tqdm(loader, desc="ResNet-200D"):
    save_heatmap(gcam_resnet.generate(imgs.to(DEVICE)), os.path.join(BINARY_OUT, 'heatmaps_resnet'), img_ids[0])

gcam_resnet.remove_hooks(); del resnet_model; gc.collect(); torch.cuda.empty_cache()


[5/5] Generating ResNet-200D Heatmaps...


ResNet-200D: 100%|██████████| 5067/5067 [25:43<00:00,  3.28it/s]


In [14]:
# ── CELL 12 : Ensemble Heatmap Fusion ────────────────────────────────────────
print("\nFusing Ensemble Heatmaps...")
all_ids = [Path(f).stem for f in os.listdir(os.path.join(BINARY_OUT, 'heatmaps_densenet')) if f.endswith('.npy')]
index_rows = []

for iid in tqdm(all_ids, desc="Fusing"):
    maps = []
    # FIXED: Added missing comma between swin and convnext
    for d in ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin', 'heatmaps_convnext', 'heatmaps_resnet']:
        p = os.path.join(BINARY_OUT, d, f'{iid}.npy')
        if os.path.exists(p):
            m = np.load(p).astype(np.float32)
            m = upsample_heatmap(m, 224, 224)
            if m.max() > 0:
                m = (m - m.min()) / (m.max() - m.min())
            maps.append(m)
    
    if maps:
        fused = np.mean(maps, axis=0)
        fused = (fused - fused.min()) / (fused.max() - fused.min() + 1e-8)
        save_heatmap(fused, os.path.join(BINARY_OUT, 'heatmaps_ensemble'), iid)
        
        index_rows.append({
            'image_id': iid,
            'densenet_hmap': os.path.join(BINARY_OUT, 'heatmaps_densenet', f'{iid}.npy'),
            'efficientnet_hmap': os.path.join(BINARY_OUT, 'heatmaps_efficientnet', f'{iid}.npy'),
            'swin_hmap': os.path.join(BINARY_OUT, 'heatmaps_swin', f'{iid}.npy'),
            'convnext_hmap': os.path.join(BINARY_OUT, 'heatmaps_convnext', f'{iid}.npy'),
            'resnet_hmap': os.path.join(BINARY_OUT, 'heatmaps_resnet', f'{iid}.npy'),
            'ensemble_hmap': os.path.join(BINARY_OUT, 'heatmaps_ensemble', f'{iid}.npy')
        })

pd.DataFrame(index_rows).to_csv(os.path.join(BINARY_OUT, 'heatmap_index.csv'), index=False)
print("Heatmap generation and fusion complete.")


Fusing Ensemble Heatmaps...


Fusing: 100%|██████████| 5067/5067 [00:17<00:00, 289.20it/s]

Heatmap generation and fusion complete.
